# RAG и делегирование поиска на LangGraph

Два агента с фиксированными ролями. **Менеджер** отправляет задание **Поисковику**, получает источники и возвращает ответ пользователю. Обмен выполняется через структурированные сообщения в состоянии LangGraph.

Пример запускается без токенов и внешних серверов. Используются синтетические бухгалтерские документы, локальный Qdrant, векторизация TF-IDF и лексический реранкинг. Связи документов хранятся в памяти. Итоговый ответ собирается из цитат без вызова LLM. Схема подключения LLM приведена в конце.

В полном RAG-контуре используются нейросетевые эмбеддинги, Qdrant, граф связей Neo4j и LLM. Здесь упрощены модели и хранилища, а делегирование, поиск и проверка доступа выполняются.


In [1]:
%pip install -q "langgraph==1.2.12" "qdrant-client==1.18.0"


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 1. Документы и чанкинг

Размер фрагмента - до 1800 символов, шаг - 1600. Перекрытие соседних полных фрагментов составляет 200 символов. Границы страниц сохраняются. Каждый фрагмент содержит ID документа, страницу, организацию, группы доступа, статус и подтверждённый ключ договора.
Короткие документы примера помещаются в один фрагмент.


In [1]:
from __future__ import annotations

import math
import operator
import re
import uuid
from collections import Counter
from typing import Annotated, Literal, TypedDict

from langgraph.graph import END, START, StateGraph
from langgraph.types import Command
from qdrant_client import QdrantClient, models

DOCUMENTS = [
    {"document_id": "contract-42", "filename": "Договор-42.txt", "page": 1,
     "text": "Договор № 42 от 01.09.2026. ИНН контрагента 7700000000. Оплата услуг производится по актам.",
     "tenant": "demo", "acl": ["shared"], "status": "ready", "contract_key": "42|2026-09-01|7700000000"},
    {"document_id": "act-42", "filename": "Акт-1.txt", "page": 1,
     "text": "Акт № 1 к договору № 42 от 01.09.2026. ИНН 7700000000. Услуги оказаны на сумму 10000.00 рублей.",
     "tenant": "demo", "acl": ["shared"], "status": "ready", "contract_key": "42|2026-09-01|7700000000"},
    {"document_id": "payment-42", "filename": "Платеж-17.txt", "page": 1,
     "text": "Платёжное поручение № 17. Оплата по договору № 42 от 01.09.2026. ИНН 7700000000. Сумма 6000.00 рублей.",
     "tenant": "demo", "acl": ["finance"], "status": "ready", "contract_key": "42|2026-09-01|7700000000"},
    {"document_id": "payment-99", "filename": "Платеж-99.txt", "page": 1,
     "text": "Платёжное поручение № 99. Оплата по договору № 99. Сумма 3000.00 рублей.",
     "tenant": "demo", "acl": ["shared"], "status": "ready", "contract_key": "99|2026-09-02|7700000000"},
    {"document_id": "other-company", "filename": "Другой-клиент.txt", "page": 1,
     "text": "Платёжное поручение № 25 другой организации. Оплата по договору № 42. Сумма 8500.00 рублей.",
     "tenant": "other", "acl": ["shared", "finance"], "status": "ready", "contract_key": "42|2026-09-01|7700000000"},
    {"document_id": "unfinished", "filename": "Не-готов.txt", "page": 1,
     "text": "Платёжное поручение № 26. Оплата по договору № 42. Сумма 2500.00 рублей. Документ находится на обработке.",
     "tenant": "demo", "acl": ["shared"], "status": "indexing", "contract_key": "42|2026-09-01|7700000000"},
]

def chunk_pages(pages, size=1800, step=1600):
    assert 0 < step <= size
    chunks = []
    for page in pages:
        for start in range(0, len(page["text"]), step):
            text = page["text"][start:start + size]
            chunk_id = str(uuid.uuid5(uuid.NAMESPACE_URL,
                f'{page["tenant"]}:{page["document_id"]}:{page["page"]}:{start}'))
            chunks.append({**page, "id": chunk_id, "text": text, "start": start})
    return chunks

CHUNKS = chunk_pages(DOCUMENTS)
REPOSITORY = {chunk["id"]: chunk for chunk in CHUNKS}
print(f"Документов: {len(DOCUMENTS)}; фрагментов: {len(CHUNKS)}")

Документов: 6; фрагментов: 6


## 2. Векторизация и Vector DB

TF-IDF создаёт числовой вектор по словам текста. Это упрощённая лексическая векторизация. Для семантического поиска в полном контуре применяется локальная модель `paraphrase-multilingual-MiniLM-L12-v2` с размерностью 384.

Qdrant запускается в памяти процесса. Поиск выполняется по косинусному сходству с фильтрами организации, групп доступа и готовности документа.


In [2]:
def tokenize(text):
    tokens = re.findall(r"[а-яa-z0-9]+", text.lower().replace("ё", "е"))
    return ["платеж" if t.startswith("платеж") else
            "договор" if t.startswith("договор") else t for t in tokens]

VOCAB = sorted({token for chunk in CHUNKS for token in tokenize(chunk["text"])})
POSITIONS = {token: i for i, token in enumerate(VOCAB)}
IDF = {token: math.log((1 + len(CHUNKS)) /
       (1 + sum(token in tokenize(c["text"]) for c in CHUNKS))) + 1 for token in VOCAB}

def embed(text):
    counts = Counter(tokenize(text))
    vector = [0.0] * len(VOCAB)
    for token, count in counts.items():
        if token in POSITIONS:
            vector[POSITIONS[token]] = (1 + math.log(count)) * IDF[token]
    length = math.sqrt(sum(value * value for value in vector))
    return [value / length for value in vector] if length else vector

client = QdrantClient(location=":memory:")
COLLECTION = "documents"
client.create_collection(COLLECTION,
    vectors_config=models.VectorParams(size=len(VOCAB), distance=models.Distance.COSINE))
client.upsert(COLLECTION, points=[models.PointStruct(
    id=c["id"], vector=embed(c["text"]), payload=c) for c in CHUNKS])
print("Qdrant готов. Размерность векторов:", len(VOCAB))

Qdrant готов. Размерность векторов: 40


## 3. Поиск, связи документов и реранкинг

Сначала находятся кандидаты в Qdrant. Если назван номер договора, поиск ограничивается соответствующими ключами договора. Затем добавляются доступные фрагменты того же договора. Это минимальное представление графового расширения. В полном контуре связи хранит Neo4j.

Реранкинг повторно сортирует кандидатов по совпадениям значимых слов вопроса, затем по векторной оценке. Номер договора проверяется фильтром, а не только сходством текста. Это лексический реранкинг, без cross-encoder. Права и статус источников повторно проверяются по исходным метаданным.


In [3]:
STOP_WORDS = {"по", "и", "в", "на", "к", "от", "найди", "какие", "документы", "покажи"}

def allowed(chunk, principal):
    return (chunk["tenant"] == principal["tenant"] and chunk["status"] == "ready"
            and bool(set(chunk["acl"]) & set(principal["groups"])))

def contract_number(question):
    match = re.search(r"договор\w*\s*(?:№|номер)?\s*(\d+)", question, re.I)
    return match.group(1) if match else None

def lexical_score(question, text):
    terms = set(tokenize(question)) - STOP_WORDS
    return len(terms & set(tokenize(text))) / max(1, len(terms))

def rerank(question, candidates, top_k=4):
    unique = {candidate["id"]: candidate for candidate in candidates}
    return sorted(unique.values(), key=lambda c:
        (-lexical_score(question, c["text"]), -c["vector_score"], c["id"]))[:top_k]

def retrieve(question, principal, search_limit=4):
    if not principal["groups"]:
        return [], []
    number = contract_number(question)
    contract_keys = sorted({c["contract_key"] for c in CHUNKS
        if allowed(c, principal) and c["contract_key"].split("|")[0] == number}) if number else []
    if number and not contract_keys:
        return [], []
    query_vector = embed(question)
    if not any(query_vector):
        return [], []
    conditions = [
        models.FieldCondition(key="tenant", match=models.MatchValue(value=principal["tenant"])),
        models.FieldCondition(key="acl", match=models.MatchAny(any=principal["groups"])),
        models.FieldCondition(key="status", match=models.MatchValue(value="ready")),
    ]
    if number:
        conditions.append(models.FieldCondition(key="contract_key", match=models.MatchAny(any=contract_keys)))
    hits = client.query_points(COLLECTION, query=query_vector,
        query_filter=models.Filter(must=conditions), limit=search_limit, score_threshold=0.05).points
    seeds = [{**REPOSITORY[str(hit.id)], "vector_score": hit.score}
             for hit in hits if allowed(REPOSITORY[str(hit.id)], principal)]
    candidates = list(seeds)
    paths = []
    seed_ids = {s["id"] for s in seeds}
    for seed in seeds:
        for chunk in CHUNKS:
            if (chunk["id"] not in seed_ids and seed["contract_key"]
                and chunk["contract_key"] == seed["contract_key"] and allowed(chunk, principal)):
                candidates.append({**chunk, "vector_score": 0.0})
                paths.append({"seed": seed["id"], "id": chunk["id"], "contract": chunk["contract_key"]})
    context = rerank(question, candidates)
    context = [c for c in context if allowed(REPOSITORY[c["id"]], principal)]
    visible_ids = {c["id"] for c in context}
    paths = [p for p in paths if p["id"] in visible_ids and p["seed"] in visible_ids]
    return context, paths

## 4. Менеджер и Поисковик

Паттерн - менеджер с делегированием специализированному агенту. **Менеджер** управляет заданием и итоговым ответом. **Поисковик** отвечает только за поиск и возвращает источники. Он не формирует ответ пользователю.

Сообщение содержит отправителя, получателя, тип и данные. После ответа Поисковика Менеджер формирует итоговый ответ пользователю и завершает обработку запроса. Маршрутизация фиксирована и не требует решений LLM.


In [4]:
class State(TypedDict):
    question: str
    principal: dict
    messages: Annotated[list[dict], operator.add]
    context: list[dict]
    status: str
    answer: str

def message(sender, recipient, kind, payload):
    return {"sender": sender, "recipient": recipient, "type": kind, "payload": payload}

def manager(state: State) -> Command[Literal["searcher", "__end__"]]:
    if not state["messages"]:
        request = message("manager", "searcher", "search_request",
            {"question": state["question"], "principal": state["principal"]})
        return Command(update={"messages": [request]}, goto="searcher")
    response = state["messages"][-1]
    assert response["sender"] == "searcher" and response["type"] == "search_result"
    context = response["payload"]["context"]
    if any(not allowed(REPOSITORY[c["id"]], state["principal"]) for c in context):
        context, status, answer = [], "unavailable", "Доступ изменился. Повторите запрос."
    elif not context:
        status, answer = "insufficient_evidence", "Нет доступных источников для ответа."
    else:
        status = "answered"
        answer = "По доступным документам найдены следующие сведения:\n" + "\n".join(
            f'[{i}] {c["filename"]}, стр. {c["page"]}: {c["text"]}'
            for i, c in enumerate(context, 1))
    reply = message("manager", "user", "answer", {"status": status, "answer": answer})
    return Command(update={"messages": [reply], "context": context,
                           "status": status, "answer": answer}, goto=END)

def searcher(state: State) -> Command[Literal["manager"]]:
    request = state["messages"][-1]
    assert request["sender"] == "manager" and request["recipient"] == "searcher"
    context, paths = retrieve(request["payload"]["question"], request["payload"]["principal"])
    response = message("searcher", "manager", "search_result", {"context": context, "paths": paths})
    return Command(update={"messages": [response]}, goto="manager")

builder = StateGraph(State)
builder.add_node("manager", manager)
builder.add_node("searcher", searcher)
builder.add_edge(START, "manager")
workflow = builder.compile()

def ask(question, groups=("shared", "finance"), tenant="demo"):
    return workflow.invoke({"question": question,
        "principal": {"tenant": tenant, "groups": list(groups)},
        "messages": [], "context": [], "status": "", "answer": ""},
        config={"recursion_limit": 6})

## 5. Запуск и журнал сообщений

Журнал должен показать три сообщения. Задание Менеджера, результат Поисковика и ответ пользователю. В вопросе можно изменить номер договора или тип документа.


In [5]:
result = ask("Найди платежи по договору № 42")
for item in result["messages"]:
    print(f'{item["sender"]} → {item["recipient"]}: {item["type"]}')
print("\nСтатус:", result["status"])
print(result["answer"])

manager → searcher: search_request
searcher → manager: search_result
manager → user: answer

Статус: answered
По доступным документам найдены следующие сведения:
[1] Платеж-17.txt, стр. 1: Платёжное поручение № 17. Оплата по договору № 42 от 01.09.2026. ИНН 7700000000. Сумма 6000.00 рублей.
[2] Договор-42.txt, стр. 1: Договор № 42 от 01.09.2026. ИНН контрагента 7700000000. Оплата услуг производится по актам.
[3] Акт-1.txt, стр. 1: Акт № 1 к договору № 42 от 01.09.2026. ИНН 7700000000. Услуги оказаны на сумму 10000.00 рублей.


## 6. Проверки

Проверяем обмен сообщениями, выделение правильного договора, ограничения доступа, исключение неготовых документов, отсутствие источников, перекрытие фрагментов и повторное ранжирование.


In [6]:
assert [(m["sender"], m["recipient"], m["type"]) for m in result["messages"]] == [
    ("manager", "searcher", "search_request"),
    ("searcher", "manager", "search_result"),
    ("manager", "user", "answer"),
]
assert result["status"] == "answered"
assert result["context"][0]["document_id"] == "payment-42"
assert all(c["contract_key"].startswith("42|") for c in result["context"])
reader = ask("Найди платежи по договору № 42", groups=("shared",))
assert "payment-42" not in {c["document_id"] for c in reader["context"]}
assert "6000.00" not in reader["answer"]
for response in [result, reader]:
    assert all(c["tenant"] == "demo" and c["status"] == "ready" for c in response["context"])
    assert not ({"other-company", "unfinished"} & {c["document_id"] for c in response["context"]})
    search_result = next(m for m in response["messages"] if m["type"] == "search_result")
    assert not ({"other-company", "unfinished"} &
                {c["document_id"] for c in search_result["payload"]["context"]})
missing = ask("Найди платежи по договору № 999")
assert missing["status"] == "insufficient_evidence" and missing["context"] == []
assert ask("Найди платежи", groups=())["context"] == []
unknown = ask("абракадабра")
assert unknown["status"] == "insufficient_evidence"
long_page = {**DOCUMENTS[0], "text": "А" * 1600 + "Б" * 200 + "В" * 1200}
parts = chunk_pages([long_page])
assert parts[0]["text"][-200:] == parts[1]["text"][:200]
assert [p["start"] for p in parts] == [0, 1600]
ranked = rerank("платеж", [
    {"id": "a", "text": "договор услуги", "vector_score": 0.9},
    {"id": "b", "text": "платеж", "vector_score": 0.1},
])
assert ranked[0]["id"] == "b"
principal = {"tenant": "demo", "groups": ["shared", "finance"]}
source = result["context"][0]
original_acl = REPOSITORY[source["id"]]["acl"]
REPOSITORY[source["id"]]["acl"] = ["revoked"]
try:
    reply = manager({"question": "платеж", "principal": principal,
        "messages": [message("searcher", "manager", "search_result", {"context": [source], "paths": []})]})
    assert reply.update["status"] == "unavailable" and reply.update["context"] == []
finally:
    REPOSITORY[source["id"]]["acl"] = original_acl
expanded, paths = retrieve("Найди платежи по договору № 42", principal, search_limit=1)
assert paths and len(expanded) > 1
assert all(path["contract"].startswith("42|") for path in paths)
assert "payment-42" in {c["document_id"] for c in expanded}
print("Все проверки пройдены, включая расширение графового контекста.")

Все проверки пройдены, включая расширение графового контекста.


## 7. Подключение LLM

После поиска вопрос и доступные фрагменты документов передаются LLM для подготовки ответа с цитатами. Перед выдачей проверяются ссылки на источники и права пользователя. В этом notebook ответ собирается из найденных цитат без вызова LLM.
